In [1]:
import sys
from dataclasses import dataclass
from operator import methodcaller

import numpy as np
import polars as pl
from funcy import first, walk_values
from scipy.integrate import solve_ivp
from tqdm.auto import trange

sys.path.append("/app")
from src.polars_utils import random_split

### Preparation

In [2]:
@dataclass
class PendulumParameters:
    """Parameter class for Pendulum simulation."""

    st: float = 0  # Start time (s)
    et_min: float = 3  # End time (s)
    et_max: float = 5  # End time (s)
    ts: float = 0.1  # Time step (s)
    g: float = 9.81  # Acceleration due to gravity (m/s^2)
    m: float = 1  # Mass of bob (kg)
    l_min: float = 0.5
    l_max: float = 10
    mean_n: int = 100  # mean length of sequence
    alpha: float = 0.5

In [3]:
# data generation from https://arxiv.org/abs/2401.15935


def hawkes_intensity(mu, alpha, points, t):
    """Find the hawkes intensity.

    The formula is as follows:
        mu + alpha * sum( np.exp(-(t-s)) for s in points if s<=t ).
    """
    p = np.array(points)
    p = p[p <= t]
    p = np.exp(p - t) * alpha
    return mu + np.sum(p)
    # return mu + alpha * sum( np.exp(s - t) for s in points if s <= t )


def simulate_hawkes(mu, alpha, st, et):
    """Simulate a Hawkes process."""
    t = st
    points = []
    all_samples = []
    while t < et:
        m = hawkes_intensity(mu, alpha, points, t)
        s = 0

        # Float32 precision will cast these to
        # same timestamp
        while s < 1e-5:
            s = np.random.exponential(scale=1 / m)

        ratio = hawkes_intensity(mu, alpha, points, t + s) / m
        if t + s >= et:
            break
        if ratio >= np.random.uniform():
            points.append(t + s)
        all_samples.append(t + s)
        t = t + s

    return np.array(points), all_samples


def create_sample(params: PendulumParameters):
    """Generate a Pendulum dataset sample.

    inspired by https://skill-lync.com/student-projects/Simulation-of-a-Simple-Pendulum-on-Python-95518.
    """
    # main

    length = np.random.uniform(params.l_min, params.l_max)
    theta1_ini = np.random.uniform(0, 2 * np.pi)  # Initial angular displacement (rad)
    theta2_ini = np.random.uniform(-np.pi, np.pi)  # Initial angular velocity (rad/s)
    theta_ini = [theta1_ini, theta2_ini]

    et = np.random.uniform(params.et_min, params.et_max)
    mu = params.mean_n * (1 - params.alpha) / (et - params.st - 1)

    b = np.random.uniform(1, 3)

    t_span = [params.st, et + params.ts]

    points, _ = simulate_hawkes(mu, params.alpha, params.st, et)
    if len(points) < 5:
        points = np.linspace(params.st, et, 5)
    t_ir = points

    def sim_pen_eq(_, theta):
        dtheta2_dt = (-b / params.m) * theta[1] + (-params.g / length) * np.sin(
            theta[0]
        )
        dtheta1_dt = theta[1]
        return [dtheta1_dt, dtheta2_dt]

    theta12 = solve_ivp(sim_pen_eq, t_span, theta_ini, t_eval=t_ir)
    theta1 = theta12.y[0, :]

    # return x, y
    # or we could return angles ...
    x = np.sin(theta1)
    y = -np.cos(theta1)

    return {"x": x, "y": y}, t_ir, b


def noise_sequence(sequence: dict):
    """Noise the input sequence."""
    L = len(first(sequence.values()))
    N = max(int((L - 1) * 0.1), 0)

    for v in sequence.values():
        v[np.random.choice(L - 1, size=N, replace=False) + 1] = np.nan
    return sequence


def create_dataset(size: int, params: PendulumParameters):
    """Create a Pendulum dataset of the specified size."""
    data = []
    for _ in trange(size):
        sequence, time, target = create_sample(params)
        sequence = noise_sequence(sequence)
        sequence = walk_values(methodcaller("tolist"), sequence)
        data.append({"target": target, "time": time.tolist()} | sequence)

    df = pl.DataFrame(data).with_columns(
        pl.col("time").list.eval(
            pl.duration(nanoseconds=pl.first() * 10**9) / pl.duration(seconds=4)
        )
    )

    df = random_split(df, train=0.7, val=0.15, test=0.15)
    return df

### Generation

In [4]:
size = 32768  # The original dataset size

The original pendulum dataset:

In [5]:
df = create_dataset(size, PendulumParameters())
df.write_parquet("/mnt/data/preprocessed/pendulum.parquet")

  0%|          | 0/32768 [00:00<?, ?it/s]

The smaller pendulum_small dataset:

In [7]:
df = create_dataset(size // 32, PendulumParameters())
df.write_parquet("/mnt/data/preprocessed/pendulum_small.parquet")


  0%|          | 0/1024 [00:00<?, ?it/s]